# Netflix Data Analysis

A reproducible exploratory data analysis of Netflix movies and TV shows. This notebook covers data quality, catalog composition, release trends, genre popularity, ratings, movie duration, and country-wise insights.

## 1. Setup and data loading

The source file is expected at `../data/netflix_titles.csv`. Run `../fetch_dataset.sh` first if the file is not present.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

ROOT = Path.cwd().parent
sys.path.append(str(ROOT / 'src'))
from analyze_netflix import load_and_clean, run

sns.set_theme(style='whitegrid', context='notebook')
df = load_and_clean(ROOT / 'data' / 'netflix_titles.csv')
df.head()

## 2. Data quality and structure

In [ ]:
print('Shape:', df.shape)
display(df.dtypes.to_frame('dtype'))
display(df.isna().sum().sort_values(ascending=False).to_frame('missing_values'))

The dataset contains both single-value and multi-value fields. The analysis keeps the raw fields and derives `primary_country` and `genre` for straightforward frequency comparisons. Missing dates and ratings are retained as unknown rather than silently discarded.

## 3. Catalog composition

In [ ]:
display(df['type'].value_counts().rename_axis('type').to_frame('titles'))
sns.countplot(data=df, x='type', palette={'Movie':'#e50914','TV Show':'#221f1f'})
plt.title('Netflix catalog mix by format', weight='bold')
plt.xlabel('Format'); plt.ylabel('Titles'); plt.show()

## 4. Release trends

In [ ]:
trend = df.groupby(['release_year','type']).size().unstack(fill_value=0)
trend.tail(40).plot(figsize=(11,5), marker='o', color=['#e50914','#221f1f'])
plt.title('Recent catalog releases by format', weight='bold')
plt.xlabel('Release year'); plt.ylabel('Titles'); plt.show()

## 5. Genre popularity

In [ ]:
genre_counts = df['listed_in'].dropna().str.split(', ').explode().value_counts()
display(genre_counts.head(15).to_frame('titles'))
genre_counts.head(12).sort_values().plot.barh(figsize=(10,6), color='#b20710')
plt.title('Most common Netflix genres', weight='bold'); plt.xlabel('Titles'); plt.show()

## 6. Ratings distribution

In [ ]:
rating_counts = df['rating'].value_counts()
display(rating_counts.to_frame('titles'))
rating_counts.head(12).sort_values().plot.barh(figsize=(10,6), color='#564d4d')
plt.title('Content rating distribution', weight='bold'); plt.xlabel('Titles'); plt.show()

## 7. Country-wise insights

In [ ]:
country_counts = df['primary_country'].replace('Unknown', np.nan).dropna().value_counts()
display(country_counts.head(15).to_frame('titles'))
country_counts.head(12).sort_values().plot.barh(figsize=(10,6), color='#e87c03')
plt.title('Leading countries by title count', weight='bold'); plt.xlabel('Titles'); plt.show()

## 8. Movie duration

In [ ]:
movies = df.loc[df['type'].eq('Movie') & df['duration_num'].notna()]
print('Median movie duration:', movies['duration_num'].median(), 'minutes')
sns.histplot(movies['duration_num'], bins=30, kde=True, color='#e50914')
plt.axvline(movies['duration_num'].median(), color='#221f1f', linestyle='--', label='Median')
plt.title('Movie duration distribution', weight='bold'); plt.xlabel('Minutes'); plt.legend(); plt.show()

## 9. Reproduce all portfolio outputs

The production script writes all charts and summary CSV files to `../outputs/`. This keeps the notebook exploratory while making the final artifacts reproducible from a single command.

In [ ]:
summary = run(df)
display(summary)

## Conclusion

This snapshot is dominated by movies, with international movies and dramas forming the largest genre groupings. TV-MA and TV-14 are the most frequent ratings, while the United States and India are the leading primary-country labels. The median movie duration is 98 minutes. These are descriptive results from a historical catalog snapshot, not current availability or causal conclusions.